In [13]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "haun2009great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Haun_Cognition_2009_APES_TRIALS.sav")
complete_path_2 = os.path.join(original_data_pathway, "BORNEO.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [14]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
# original_data_pathway_out = os.path.join(original_data_pathway, 'file name.csv')
# df.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)



In [15]:
df['study_id']="haun2009great"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns
df.rename(columns={"name": "ape",
    "species":"species_temp",
    "sex":"sex_temp",
    "alignement": "alignment"}, inplace=True)

In [16]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [17]:
df['date'] = df['date'].astype(str)
df[['year','month', 'day']] = df['date'].str.split('-',expand=True)

In [18]:
code_list = ['condition']
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == '0.0':
            entry = "3"
        elif entry =='tubes':
            entry = "2"
        elif entry =='lines':
            entry = "2"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': 'experiment'})

In [19]:

df['cuppos'].replace('\+', '_', inplace=True, regex=True)

df['choicepos'].replace('\+', '_', inplace=True, regex=True)

df.replace(' ', '_', inplace=True, regex=True)



In [20]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='ape', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365
df['age_in_months'] = df['dodc'].dt.to_period('M') - df['dob'].dt.to_period('M')
df['age_in_months'] = df['age_in_months'].apply(lambda x: x.n)

In [21]:
df.rename(columns={"ape": "participant"}, inplace=True)
# df.columns

df1 = pd.read_csv(complete_path_2)

df1['study_id']="haun2009great"
df1['experiment']="4"
df1.columns = map(str.lower, df1.columns)
df1=df1.applymap(lambda s: s.lower() if type(s) == str else s)
# df1.columns

In [22]:

df1.rename(columns={"name": "participant",
    "age_at_arrival":"age_in_years",
    "cupchoice":"choice",
    "alingement":"alignment"}, inplace=True)
df1['species']="orangutan"
df1['location']='pasir_panjang'

df1['date_of_arrival'] = df1['date_of_arrival'].astype(str)
df1[['month', 'day','year']] = df1['date_of_arrival'].str.split('/',expand=True)
df1['sex'].replace('female', 'f', inplace=True, regex=True)
df1['sex'].replace('male', 'm', inplace=True, regex=True)

# df1 = df1[[]]
# .columns
data_frames=[df,df1]

fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

fulldf.rename(columns={"firstcond": "first_condition",
    "cup":"correct_cup",
    "cuppos":"correct_cup_position",
    "choicepos":"chosen_cup_position"}, inplace=True)


fulldf['condition'].replace('0.0', 'nothing', inplace=True, regex=True)
fulldf['first_condition'].replace(0, 'nothing', inplace=True, regex=True)
# fulldf['condition'].unique()

fulldf.loc[fulldf.experiment == '3', ['first_condition']] = np.nan

In [23]:
fulldf=fulldf[['study_id','experiment','year', 'month', 'day','date_of_arrival',
        'participant',  'age_in_years', 'sex','species',  'genus', 'location', 
        'session',   'trial','condition', 'first_condition','alignment', 'correct_cup', 'correct_cup_position',
       'choice', 'chosen_cup_position', 'correct']]



In [24]:
for index in range(2,5):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'haun2009great_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'haun2009great_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
